# One DFS, three answers - Tarjan's timestamps

Depth-first search visits every vertex once. If you write down two numbers while it runs,
the same traversal answers three different structural questions:

| | meaning |
|---|---|
| `disc[u]` | the moment the DFS first reached `u` - a clock that only increases |
| `low[u]`  | the smallest `disc` reachable from `u`'s subtree via tree edges plus at most one backward edge |

`low` answers one question - *can this subtree get above me without going through me?* -
and the three classics are three readings of it:

| test | meaning | graph |
|---|---|---|
| `low[v] > disc[u]` | the tree edge `u-v` is a **bridge** | undirected |
| `low[v] >= disc[u]` | `u` is an **articulation point** | undirected |
| `low[u] == disc[u]` | `u` is the root of a **strongly connected component** | directed |

In [1]:
# Undirected: two triangles glued at vertex 2, one edge across to a third triangle.
UN_N = 8
UN_EDGES = [(0, 1), (1, 2), (2, 0),
            (2, 3), (3, 4), (4, 2),
            (4, 5),
            (5, 6), (6, 7), (7, 5)]

# Directed: SCCs are {1,2,3}, {4,5} and {0}.  The edge 5->2 is the trap.
DI_N = 6
DI_EDGES = [(0, 1), (0, 4),
            (1, 2), (2, 3), (3, 1),
            (4, 5), (5, 4),
            (5, 2)]

def undirected_adj(n, edges):
    """(neighbour, edge_id) - the id is what makes the parent test correct."""
    adj = [[] for _ in range(n)]
    for i, (u, v) in enumerate(edges):
        adj[u].append((v, i))
        adj[v].append((u, i))
    return adj

def directed_adj(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
    return adj

def show_times(disc, low):
    print('vertex ' + ' '.join('%4d' % i for i in range(len(disc))))
    print('disc   ' + ' '.join('%4d' % d for d in disc))
    print('low    ' + ' '.join('%4d' % l for l in low))

print('undirected:', UN_EDGES)
print('directed  :', DI_EDGES)

undirected: [(0, 1), (1, 2), (2, 0), (2, 3), (3, 4), (4, 2), (4, 5), (5, 6), (6, 7), (7, 5)]
directed  : [(0, 1), (0, 4), (1, 2), (2, 3), (3, 1), (4, 5), (5, 4), (5, 2)]


## 1. Bridges

A tree edge `u -> v` is a bridge when `v`'s subtree has no way back up. "Up" means a
vertex discovered before `u`, so the test is `low[v] > disc[u]`: strictly above, not
equal, because reaching `u` itself still means going through the edge we are testing.

Note the parameter: we skip the **edge** we arrived on, not the parent vertex.

In [2]:
def find_bridges(n, edges):
    adj = undirected_adj(n, edges)
    disc = [-1] * n
    low = [0] * n
    clock = 0
    bridges = []

    def dfs(u, in_edge):
        nonlocal clock
        disc[u] = low[u] = clock
        clock += 1
        for v, eid in adj[u]:
            if eid == in_edge:          # skip the edge we came in on
                continue
            if disc[v] == -1:           # tree edge
                dfs(v, eid)
                low[u] = min(low[u], low[v])
                if low[v] > disc[u]:
                    bridges.append((u, v))
            else:                       # back edge to an ancestor
                low[u] = min(low[u], disc[v])

    for s in range(n):
        if disc[s] == -1:
            dfs(s, -1)
    return bridges, disc, low

bridges, disc, low = find_bridges(UN_N, UN_EDGES)
show_times(disc, low)
print('bridges:', bridges)

vertex    0    1    2    3    4    5    6    7
disc      0    1    2    3    4    5    6    7
low       0    0    0    2    2    5    5    5
bridges: [(4, 5)]


## 2. Articulation points

Same DFS, one character different. A bridge asks whether the *edge* is essential; a cut
vertex asks whether the *vertex* is. If `v`'s subtree can climb back exactly as far as
`u` but no further, the edge is not critical but `u` still is - hence `>=`.

The root has no parent to be cut from, so it gets its own rule: it is a cut vertex
exactly when the DFS started more than one subtree from it.

In [3]:
def find_articulation_points(n, edges):
    adj = undirected_adj(n, edges)
    disc = [-1] * n
    low = [0] * n
    clock = 0
    cut = set()

    def dfs(u, in_edge, root):
        nonlocal clock
        disc[u] = low[u] = clock
        clock += 1
        children = 0
        for v, eid in adj[u]:
            if eid == in_edge:
                continue
            if disc[v] == -1:
                children += 1
                dfs(v, eid, root)
                low[u] = min(low[u], low[v])
                if u != root and low[v] >= disc[u]:
                    cut.add(u)
            else:
                low[u] = min(low[u], disc[v])
        if u == root and children > 1:
            cut.add(u)

    for s in range(n):
        if disc[s] == -1:
            dfs(s, -1, s)
    return sorted(cut)

cuts = find_articulation_points(UN_N, UN_EDGES)
print('bridges         :', bridges)
print('cut vertices    :', cuts)
print()
print('4 and 5 are the endpoints of the bridge.  2 is on no bridge at all:')
print('it is where the two triangles are glued, so no single edge is critical,')
print('but the vertex is.')

bridges         : [(4, 5)]
cut vertices    : [2, 4, 5]

4 and 5 are the endpoints of the bridge.  2 is on no bridge at all:
it is where the two triangles are glued, so no single edge is critical,
but the vertex is.


## 3. Why the parent *edge* and not the parent *vertex*

Skipping `v == parent` looks equivalent and is - until two vertices are joined twice.
Two cables between the same pair of routers make the link redundant, but the buggy
version skips *both* cables and never notices the alternative route.

In [4]:
def find_bridges_by_parent_vertex(n, edges):
    adj = undirected_adj(n, edges)
    disc = [-1] * n
    low = [0] * n
    clock = 0
    bridges = []

    def dfs(u, parent):
        nonlocal clock
        disc[u] = low[u] = clock
        clock += 1
        for v, _eid in adj[u]:
            if v == parent:             # <-- the bug
                continue
            if disc[v] == -1:
                dfs(v, u)
                low[u] = min(low[u], low[v])
                if low[v] > disc[u]:
                    bridges.append((u, v))
            else:
                low[u] = min(low[u], disc[v])

    for s in range(n):
        if disc[s] == -1:
            dfs(s, -1)
    return bridges

doubled = UN_EDGES + [(4, 5)]                       # a second, parallel 4-5 link
print('edge-id version  :', find_bridges(UN_N, doubled)[0], ' <- redundant now')
print('parent-vertex ver:', find_bridges_by_parent_vertex(UN_N, doubled),
      ' <- still critical')

edge-id version  : []  <- redundant now
parent-vertex ver: [(4, 5)]  <- still critical


## 4. Strongly connected components

In a directed graph, "connected" splits in two: `u` reaches `v` and `v` reaches `u`.
An SCC is a maximal set where that holds for every pair.

The DFS keeps every vertex it has not yet assigned on a stack. When a vertex finishes
with `low[u] == disc[u]`, nothing in its subtree found a way above it, so `u` is the
root of a component - and the component is exactly what sits above `u` on the stack.

In [5]:
def tarjan_scc(n, edges):
    adj = directed_adj(n, edges)
    disc = [-1] * n
    low = [0] * n
    on_stack = [False] * n
    stack, comps = [], []
    clock = 0

    def dfs(u):
        nonlocal clock
        disc[u] = low[u] = clock
        clock += 1
        stack.append(u)
        on_stack[u] = True
        for v in adj[u]:
            if disc[v] == -1:
                dfs(v)
                low[u] = min(low[u], low[v])
            elif on_stack[v]:            # only an ancestor counts
                low[u] = min(low[u], disc[v])
            # else: v is in a component that is already finished - ignore
        if low[u] == disc[u]:
            comp = []
            while True:
                w = stack.pop()
                on_stack[w] = False
                comp.append(w)
                if w == u:
                    break
            comps.append(sorted(comp))

    for s in range(n):
        if disc[s] == -1:
            dfs(s)
    return comps, disc, low

comps, ddisc, dlow = tarjan_scc(DI_N, DI_EDGES)
show_times(ddisc, dlow)
print('components:', comps)

vertex    0    1    2    3    4    5
disc      0    1    2    3    4    5
low       0    1    1    1    4    4
components: [[1, 2, 3], [4, 5], [0]]


## 5. The on-stack test is the whole algorithm

Drop `elif on_stack[v]` and the code still runs, still terminates, still prints
components. The edge `5 -> 2` points into `{1,2,3}`, which finished long ago; its `disc`
is small, so relaxing `low` against it drags 4, 5 and even 0 into one blob.

*On the stack* is how the algorithm distinguishes an ancestor from a stranger.

In [6]:
def tarjan_scc_without_onstack(n, edges):
    adj = directed_adj(n, edges)
    disc = [-1] * n
    low = [0] * n
    stack, comps = [], []
    clock = 0

    def dfs(u):
        nonlocal clock
        disc[u] = low[u] = clock
        clock += 1
        stack.append(u)
        for v in adj[u]:
            if disc[v] == -1:
                dfs(v)
                low[u] = min(low[u], low[v])
            else:
                low[u] = min(low[u], disc[v])     # <-- no on_stack test
        if low[u] == disc[u]:
            comp = []
            while True:
                w = stack.pop()
                comp.append(w)
                if w == u:
                    break
            comps.append(sorted(comp))

    for s in range(n):
        if disc[s] == -1:
            dfs(s)
    return comps

print('correct:', comps)
print('buggy  :', tarjan_scc_without_onstack(DI_N, DI_EDGES))

correct: [[1, 2, 3], [4, 5], [0]]
buggy  : [[1, 2, 3], [0, 4, 5]]


## 6. Condensation - every directed graph is a DAG of its SCCs

Collapse each component to a single vertex and the result is acyclic, by construction:
a cycle between two components would have merged them. That is what makes "topological
order" a sensible request for a graph that contains cycles, and it is the standard first
step of 2-SAT, of dependency resolution with circular imports, and of the DAG scheduling
we did with Kahn's algorithm.

In [7]:
from collections import defaultdict

def condensation(n, edges, comps):
    cid = [0] * n
    for i, comp in enumerate(comps):
        for u in comp:
            cid[u] = i
    dag = defaultdict(set)
    for u, v in edges:
        if cid[u] != cid[v]:
            dag[cid[u]].add(cid[v])
    return cid, {k: sorted(v) for k, v in dag.items()}

cid, dag = condensation(DI_N, DI_EDGES, comps)
print('component of each vertex:', cid)
print('edges between components:', dag)
print()
print('Tarjan emits components in reverse topological order of this DAG,')
print('so reversing the list gives a topological order for free.')

component of each vertex: [2, 0, 0, 0, 1, 1]
edges between components: {2: [0, 1], 1: [0]}

Tarjan emits components in reverse topological order of this DAG,
so reversing the list gives a topological order for free.


## 7. LeetCode 1192 - Critical Connections in a Network

*"Return all connections that, if removed, will make some server unreachable."* That is
the definition of a bridge, so the algorithm is already written.

The interview-relevant part is the recursion: `n` goes up to 10^5 and CPython recurses
only 1000 deep by default, so a long chain of servers raises `RecursionError`. The
iterative rewrite keeps an explicit stack of `(vertex, incoming edge, cursor)`; the
cursor is an iterator, which is what remembers where each frame left off.

In [8]:
def critical_connections(n, connections):
    adj = [[] for _ in range(n)]
    for i, (u, v) in enumerate(connections):
        adj[u].append((v, i))
        adj[v].append((u, i))

    disc = [-1] * n
    low = [0] * n
    clock = 0
    out = []

    for s in range(n):
        if disc[s] != -1:
            continue
        disc[s] = low[s] = clock
        clock += 1
        stack = [(s, -1, iter(adj[s]))]
        while stack:
            u, in_edge, it = stack[-1]
            descended = False
            for v, eid in it:                 # the cursor resumes where it stopped
                if eid == in_edge:
                    continue
                if disc[v] == -1:
                    disc[v] = low[v] = clock
                    clock += 1
                    stack.append((v, eid, iter(adj[v])))
                    descended = True
                    break                     # go deeper
                low[u] = min(low[u], disc[v])
            if not descended:                 # u is finished - the "return"
                stack.pop()
                if stack:
                    p = stack[-1][0]
                    low[p] = min(low[p], low[u])
                    if low[u] > disc[p]:
                        out.append([p, u])
    return out

print(critical_connections(4, [[0, 1], [1, 2], [2, 0], [1, 3]]))
print(critical_connections(UN_N, [list(e) for e in UN_EDGES]))
chain = [[i, i + 1] for i in range(3000)]     # the recursive version dies here
print('3000-server chain ->', len(critical_connections(3001, chain)), 'critical links')

[[1, 3]]
[[4, 5]]
3000-server chain -> 3000 critical links


## Tests

In [9]:
assert sorted(tuple(sorted(b)) for b in bridges) == [(4, 5)]
assert cuts == [2, 4, 5]
assert find_bridges(UN_N, doubled)[0] == []
assert find_bridges_by_parent_vertex(UN_N, doubled) == [(4, 5)]
assert sorted(comps) == [[0], [1, 2, 3], [4, 5]]
assert comps[0] == [1, 2, 3]                      # deepest component finishes first
assert tarjan_scc_without_onstack(DI_N, DI_EDGES) == [[1, 2, 3], [0, 4, 5]]
assert dag == {2: [0, 1], 1: [0]}
assert critical_connections(4, [[0, 1], [1, 2], [2, 0], [1, 3]]) == [[1, 3]]
assert len(critical_connections(3001, chain)) == 3000
print('all assertions passed')

all assertions passed
